# Lab 17: Retrieval evaluation and the isolation test

*Project LANTERN, Tool Tutorial Part 2. Classroom sample: Project Wren (`deal_a`) and its rival Project Heron (`deal_b`). Fictional, for DAMG 7245.*

In [1]:
import os, sys, json
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "params.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
from src.llm import client as llm
from src import schemas, prompts, sample_data as sd
print("repository:", ROOT, "| LLM mode:", llm.MODE)

repository: /Users/danysigha/Desktop/LanternCS2 | LLM mode: replay


In [2]:
import numpy as np, shutil
from src import sample_lsa
recs = sd.records("deal_a") + sd.records("deal_b")
chunks = [{"chunk_id": f'{r["doc_id"]}:{r["block_id"]}', "deal_id": r["deal_id"], "doc_id": r["doc_id"], "tier": r["tier"],
           "block_ids": [r["block_id"]], "text": r["text"]} for r in recs]
vecs = llm.embed([c["text"] for c in chunks], sample_lsa.MODEL, compute=sample_lsa.encode).tolist()
embed = lambda text: llm.embed([text], sample_lsa.MODEL, compute=sample_lsa.encode)[0].tolist()
import chromadb                    # rerun-safe: drop the collection through the client, not the folder
try:
    chromadb.PersistentClient(path="data/index").delete_collection("lantern")
except Exception:
    pass                           # first run: nothing to delete

In [3]:
# From the slide: A vector store with scope enforced in the store
import chromadb, json

client = chromadb.PersistentClient(path="data/index")
col = client.get_or_create_collection(
    "lantern", configuration={"hnsw": {"space": "cosine"}})

col.add(ids=[c["chunk_id"] for c in chunks],
        embeddings=vecs,
        documents=[c["text"] for c in chunks],
        metadatas=[{"deal_id": c["deal_id"],
                    "doc_id": c["doc_id"],
                    "tier": c["tier"],
                    "block_ids": json.dumps(c["block_ids"])}
                   for c in chunks])

def search(qvec, scope: str, k: int = 10):
    if not scope:
        raise ValueError("scope is required")
    return col.query(query_embeddings=[qvec], n_results=k,
        where={"deal_id": {"$in": [scope, "public"]}})

In [4]:
# From the slide: BM25 plus dense, fused with reciprocal rank fusion
from rank_bm25 import BM25Okapi
tok = lambda s: s.lower().split()
bm25 = BM25Okapi([tok(c["text"]) for c in chunks])

def rrf(*rankings, k=60):
    score = {}
    for ranking in rankings:
        for rank, doc in enumerate(ranking, start=1):
            score[doc] = (score.get(doc, 0)
                          + 1 / (k + rank))
    return sorted(score, key=score.get, reverse=True)

def hybrid(query, qvec, scope, k=10, rrf_k=60):
    dense = search(qvec, scope, k=50)["ids"][0]
    s = bm25.get_scores(tok(query))
    ok = (scope, "public")         # scope BM25 too
    lex = [chunks[i]["chunk_id"]
           for i in s.argsort()[::-1]
           if chunks[i]["deal_id"] in ok][:50]
    return rrf(dense, lex, k=rrf_k)[:k]

In [5]:
# From the slide: The metrics, implemented and tested
import math

def recall_at_k(ranked, rel, k):
    return len(set(ranked[:k]) & set(rel)) / len(rel)

def mrr(ranked, rel):
    for i, d in enumerate(ranked, 1):
        if d in rel:
            return 1 / i
    return 0.0

def ndcg_at_k(ranked, grades, k):
    gain = lambda g, i: (2**g - 1) / math.log2(i + 1)
    dcg = sum(gain(grades.get(d, 0), i)
              for i, d in enumerate(ranked[:k], 1))
    ideal = sorted(grades.values(), reverse=True)[:k]
    idcg = sum(gain(g, i)
               for i, g in enumerate(ideal, 1))
    return dcg / idcg if idcg else 0.0

In [6]:
import pandas as pd
qrels = [json.loads(l) for l in open("data/eval/qrels.jsonl")]
rows = []
for q in qrels:
    rel, grades = list(q["relevant"]), q["relevant"]
    dense = search(embed(q["query"]), q["scope"], k=20)["ids"][0]
    hyb = hybrid(q["query"], embed(q["query"]), q["scope"], k=20)
    for name, ranked in (("dense", dense), ("hybrid", hyb)):
        rows.append({"retriever": name, "category": q["category"],
                     "recall@5": recall_at_k(ranked, rel, 5),
                     "recall@10": recall_at_k(ranked, rel, 10),
                     "mrr": mrr(ranked, rel), "ndcg@10": ndcg_at_k(ranked, grades, 10)})
pd.DataFrame(rows).groupby(["retriever", "category"])[["recall@5", "recall@10", "mrr", "ndcg@10"]].mean().round(3)

recall@5  recall@10    mrr  ndcg@10
retriever category                                            
dense     cross_document      1.000       1.00  1.000    0.966
          data_room_claim     1.000       1.00  1.000    0.932
          definitional        1.000       1.00  1.000    1.000
          numeric_lookup      1.000       1.00  1.000    0.920
          section_seeking     1.000       1.00  1.000    1.000
          table_seeking       1.000       1.00  1.000    1.000
hybrid    cross_document      0.750       0.75  1.000    0.913
          data_room_claim     1.000       1.00  1.000    0.932
          definitional        1.000       1.00  1.000    1.000
          numeric_lookup      0.833       1.00  0.778    0.837
          section_seeking     1.000       1.00  1.000    1.000
          table_seeking       1.000       1.00  0.500    0.631

The isolation test from the slide runs unchanged: in this repository `data/eval/qrels.jsonl` and `data/dataroom/canaries_visible.json` are where Case Study 2 puts them.

In [7]:
chunk_meta = {c["chunk_id"]: c for c in chunks}.get

In [8]:
# From the slide: The isolation test, and proving it has teeth
import json, os, pytest
ROOT = os.environ.get("LANTERN_ISOLATION_FIXTURE", "data")
QRELS = f"{ROOT}/eval/qrels.jsonl"
CANS = f"{ROOT}/dataroom/canaries_visible.json"
QUERIES = [json.loads(l) for l in open(QRELS)]
CANARIES_B = json.load(open(CANS))["deal_b"]

@pytest.mark.parametrize("q", QUERIES)
def test_deal_a_never_sees_deal_b(q):
    hits = hybrid(q["query"], embed(q["query"]),
                  scope="deal_a", k=20)
    for cid in hits:
        meta = chunk_meta(cid)
        assert meta["deal_id"] in ("deal_a", "public")
        text = meta["text"]
        assert not any(c in text for c in CANARIES_B)

In [9]:
for q in QUERIES: test_deal_a_never_sees_deal_b(q)
print("isolation test passes on", len(QUERIES), "queries")
# Teeth: remove the scope filter and watch it fail
_safe = hybrid
def hybrid(query, qvec, scope, k=10):
    return [c["chunk_id"] for c in chunks][:k] + [c["chunk_id"] for c in chunks if c["deal_id"] == "deal_b"]
try:
    for q in QUERIES: test_deal_a_never_sees_deal_b(q)
    print("NO FAILURE: the test has no teeth")
except AssertionError:
    print("with the filter removed, the test fails: it has teeth")
hybrid = _safe

isolation test passes on 12 queries
with the filter removed, the test fails: it has teeth


**Exercise 17.** Write three more qrels for the transcript before looking at results. Which retriever wins on `cross_document` queries?

**Optional strategy comparison.** The original collection uses one chunk per block. This comparison rebuilds both deals with the 400-token `block_chunks` routine from notebook 15, indexes a separate collection, maps chunk hits back to block IDs, and scores the same 12 qrels. The three-new-qrels exercise is intentionally not run.

In [10]:
def block_chunks(blocks, budget=400):
    out, cur, sec = [], [], None
    def flush():
        if cur:
            out.append({"section": sec, "block_ids": [b["block_id"] for b in cur],
                        "text": "\n".join(b["text"] for b in cur)})
    for b in blocks:
        size = sum(max(1, len(x["text"]) // 4) for x in cur)
        if cur and (b["section"] != sec or
                    size + max(1, len(b["text"]) // 4) > budget):
            flush(); cur = []
        cur.append(b); sec = b["section"]
    flush()
    return out

block_records = []
for deal_id in ("deal_a", "deal_b"):
    source = sd.records(deal_id)
    for chunk in block_chunks(source, budget=400):
        matches = [r for r in source if r["block_id"] in chunk["block_ids"]
                   and r["section"] == chunk["section"]]
        assert matches and len({r["doc_id"] for r in matches}) == 1
        doc_id = matches[0]["doc_id"]
        block_records.append({
            "chunk_id": f"{doc_id}:{'+'.join(chunk['block_ids'])}",
            "deal_id": deal_id, "doc_id": doc_id,
            "tier": min(r["tier"] for r in matches),
            "block_ids": chunk["block_ids"], "text": chunk["text"]})

block_vecs = llm.embed([c["text"] for c in block_records], sample_lsa.MODEL,
                       compute=sample_lsa.encode).tolist()
try:
    client.delete_collection("lantern_block_400")
except Exception:
    pass
block_col = client.get_or_create_collection(
    "lantern_block_400", configuration={"hnsw": {"space": "cosine"}})
block_col.add(
    ids=[c["chunk_id"] for c in block_records], embeddings=block_vecs,
    documents=[c["text"] for c in block_records],
    metadatas=[{"deal_id": c["deal_id"], "doc_id": c["doc_id"],
                "tier": c["tier"], "block_ids": json.dumps(c["block_ids"])}
               for c in block_records])

def block_search(qvec, scope, k=20):
    if not scope:
        raise ValueError("scope is required")
    return block_col.query(query_embeddings=[qvec], n_results=k,
        where={"deal_id": {"$in": [scope, "public"]}})

block_bm25 = BM25Okapi([tok(c["text"]) for c in block_records])
block_by_id = {c["chunk_id"]: c for c in block_records}

def block_hybrid(query, qvec, scope, k=20, rrf_k=60):
    dense = block_search(qvec, scope, k=50)["ids"][0]
    scores = block_bm25.get_scores(tok(query))
    allowed = (scope, "public")
    lex = [block_records[i]["chunk_id"] for i in scores.argsort()[::-1]
           if block_records[i]["deal_id"] in allowed][:50]
    return rrf(dense, lex, k=rrf_k)[:k]

def map_chunks_to_blocks(chunk_ids):
    ranked, seen = [], set()
    for chunk_id in chunk_ids:
        chunk = block_by_id[chunk_id]
        for block_id in chunk["block_ids"]:
            key = f"{chunk['doc_id']}:{block_id}"
            if key not in seen:
                seen.add(key); ranked.append(key)
    return ranked

block_rows = []
for q in qrels:
    qvec = embed(q["query"])
    rankings = {
        "dense": map_chunks_to_blocks(
            block_search(qvec, q["scope"], k=20)["ids"][0]),
        "hybrid": map_chunks_to_blocks(
            block_hybrid(q["query"], qvec, q["scope"], k=20))}
    for retriever, ranked in rankings.items():
        block_rows.append({"retriever": retriever, "category": q["category"],
            "recall@10": recall_at_k(ranked, list(q["relevant"]), 10),
            "mrr": mrr(ranked, list(q["relevant"]))})

print("block chunks:", len(block_records))
pd.DataFrame(block_rows).groupby(["retriever", "category"])[
    ["recall@10", "mrr"]].mean().round(3)

block chunks: 9


recall@10    mrr
retriever category                         
dense     cross_document       1.000  0.667
          data_room_claim      1.000  0.444
          definitional         1.000  0.500
          numeric_lookup       1.000  0.778
          section_seeking      1.000  0.750
          table_seeking        1.000  0.167
hybrid    cross_document       0.750  0.667
          data_room_claim      1.000  0.444
          definitional         1.000  0.500
          numeric_lookup       0.833  0.704
          section_seeking      1.000  0.750
          table_seeking        1.000  0.167